In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
if "snakemake" in locals():
    assert isinstance(snakemake.input, dict)
    scenarios = snakemake.input
    reference = snakemake.params.reference
else:
    scenarios = dict(baseline="/path/to/baseline",
                     gpe="/path/to/gpe")
    reference = "baseline"

assert len(scenarios) == 2

In [ ]:
dfs_trips = []
for s in scenarios:
    df_trips = pd.read_csv("%s/eqasim_trips.csv" % scenarios[s], sep=";")
    df_trips["scenario"] = s
    dfs_trips.append(df_trips)

df_trips = pd.concat(dfs_trips)
dfs_trips = []

In [ ]:
df_trips.head()

In [ ]:
df_trips.head().groupby(["person_id", "person_trip_id", "scenario"])["travel_time"].mean()

In [ ]:
df_trips.head()[["person_id", "person_trip_id", "scenario", "travel_time"]].melt(id_vars=["person_id", "person_trip_id"], value_vars="travel_time")

In [ ]:
df_comparison = df_trips.pivot_table(values="travel_time", columns="scenario", index=["person_id", "person_trip_id"]).reset_index()
df_plot = df_comparison.melt(id_vars=["person_id", "person_trip_id"], value_name="travel_time", var_name="scenario").sort_values(["person_id", "person_trip_id", "scenario"])

for c in df_comparison.columns:
    if c in ["person_id", "person_trip_id", reference]:
        continue
    df_comparison["gain_%s" % c]  = df_comparison[reference] - df_comparison[c] 
df_comparison.head()

In [ ]:
fig = px.histogram(df_plot, x="travel_time", color="scenario", barmode="overlay")
fig.show()

In [ ]:
fig = px.histogram(df_comparison, x="gain_gpe")
fig.show()